# 🎬 MovieLens — Exploratory Data Analysis

**Dataset:** MovieLens ml-latest-small | **Source:** [GroupLens Research](https://grouplens.org/datasets/movielens/)

---

Complete exploratory analysis of the MovieLens dataset (~100k ratings from ~600 users on ~9k movies).

## Objectives

- Explore the distribution of user ratings
- Analyze average ratings per movie
- Segment ratings into bins using `pd.cut`
- Compare rating profiles: **Toy Story** vs. **Jumanji**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')
plt.rcParams.update({'figure.dpi': 100})

PATH_MOVIES  = 'Material/introducao-a-data-science-aula0/aula0/ml-latest-small/movies.csv'
PATH_RATINGS = 'Material/introducao-a-data-science-aula0/aula0/ml-latest-small/ratings.csv'

## 1. Data Loading

In [ ]:
df_movies  = pd.read_csv(PATH_MOVIES)
df_ratings = pd.read_csv(PATH_RATINGS)

print(f"Movies:   {df_movies.shape[0]:,} rows x {df_movies.shape[1]} columns")
print(f"Ratings:  {df_ratings.shape[0]:,} rows x {df_ratings.shape[1]} columns")

In [ ]:
df_movies.head()

In [ ]:
df_ratings.head()

## 2. Rating Distribution

Analysis of user-assigned ratings: frequency, descriptive statistics, and visualizations.

In [ ]:
print("Count by rating value:")
print(df_ratings['rating'].value_counts().sort_index())
print()
print(f"Mean:           {df_ratings['rating'].mean():.3f}")
print(f"Median:         {df_ratings['rating'].median():.1f}")
print(f"Std Dev:        {df_ratings['rating'].std():.3f}")

In [ ]:
df_ratings['rating'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_ratings['rating'].plot(
    kind='hist', ax=axes[0],
    color='steelblue', edgecolor='white', bins=9
)
axes[0].set_title('Rating Histogram')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frequency')

sns.boxplot(x=df_ratings['rating'], ax=axes[1], color='steelblue')
axes[1].set_title('Rating Boxplot')
axes[1].set_xlabel('Rating')

plt.suptitle('Rating Distribution — MovieLens', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

> **Insight:** Most ratings are concentrated between **3.0 and 4.0**.
> The median exceeds the mean, indicating users tend to rate positively the movies they choose to watch.

## 3. Average Rating per Movie

Grouping by `movieId` to compute the average rating per title.

In [ ]:
rating_means = df_ratings.groupby('movieId')['rating'].mean()

print(f"Total rated movies: {len(rating_means):,}")
print()
print(rating_means.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(rating_means, bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Per-Movie Averages')
axes[0].set_xlabel('Average Rating')
axes[0].set_ylabel('Number of Movies')

sns.boxplot(x=rating_means, ax=axes[1], color='steelblue')
axes[1].set_title('Boxplot of Per-Movie Averages')
axes[1].set_xlabel('Average Rating')

plt.suptitle('Per-Movie Rating Averages', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

> **Insight:** The distribution of per-movie averages is smoother and centered (~3.5), showing that even movies with few ratings tend to have reasonable median scores.

## 4. Rating Segmentation (`pd.cut`)

Binning individual ratings into discrete intervals for group analysis.

In [ ]:
bins   = [0, 1, 2, 3, 4, 5]
labels = ['0-1', '1-2', '2-3', '3-4', '4-5']

rating_ranges = pd.cut(df_ratings['rating'], bins=bins, labels=labels)
range_counts  = rating_ranges.value_counts().sort_index()

print("Distribution by rating bin:")
print(range_counts.to_string())
print()
print(f"Dominant bin: '{range_counts.idxmax()}' with {range_counts.max():,} ratings")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
range_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white', rot=0)
ax.set_title('Ratings by Rating Bin')
ax.set_xlabel('Rating Bin')
ax.set_ylabel('Number of Ratings')
plt.tight_layout()
plt.show()

## 5. Movie Comparison: Toy Story vs. Jumanji

Comparative analysis of rating distributions for two classic **1995** films in the dataset.

In [ ]:
toy_story = df_ratings.query('movieId == 1')
jumanji   = df_ratings.query('movieId == 2')

print(f"Toy Story — {len(toy_story):>5,} ratings | mean = {toy_story.rating.mean():.3f} | std = {toy_story.rating.std():.3f}")
print(f"Jumanji   — {len(jumanji):>5,} ratings | mean = {jumanji.rating.mean():.3f} | std = {jumanji.rating.std():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

toy_story['rating'].plot(
    kind='hist', ax=axes[0],
    color='steelblue', edgecolor='white', bins=9
)
axes[0].set_title('Toy Story (1995)')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frequency')
axes[0].set_xlim(0, 5.5)

jumanji['rating'].plot(
    kind='hist', ax=axes[1],
    color='salmon', edgecolor='white', bins=9
)
axes[1].set_title('Jumanji (1995)')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Frequency')
axes[1].set_xlim(0, 5.5)

plt.suptitle('Rating Distribution: Toy Story vs. Jumanji', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
compare = pd.concat([
    toy_story[['rating']].assign(Movie='Toy Story'),
    jumanji[['rating']].assign(Movie='Jumanji')
])

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(
    [toy_story['rating'], jumanji['rating']],
    labels=['Toy Story', 'Jumanji'],
    patch_artist=True,
    boxprops=dict(facecolor='steelblue', color='navy'),
    medianprops=dict(color='white', linewidth=2)
)
ax.set_title('Comparative Boxplot: Toy Story vs. Jumanji')
ax.set_ylabel('Rating')
ax.yaxis.grid(True)
plt.tight_layout()
plt.show()

## Conclusions

| Aspect                        | Finding                                                        |
|------------------------------|----------------------------------------------------------------|
| Overall average rating        | ~3.5 — users tend to rate movies positively                    |
| Most popular rating bin       | **3–4** holds the majority of all ratings                      |
| Toy Story vs. Jumanji         | Toy Story shows higher average and consistency                 |
| Variability                   | Jumanji has higher spread (larger standard deviation)          |

**Suggested next steps:** genre analysis, temporal evolution of ratings, collaborative filtering recommender system.

---
*Dataset: MovieLens ml-latest-small | GroupLens Research*